# Fine-Tuning FLAN-T5-Base for Topic Labeling

**Task**: Given a list of keywords (from topic modeling) and a research field, generate an appropriate topic label.

**Strategy**:
- **Model**: `google/flan-t5-base` (250M params) — instruction-tuned seq2seq, ideal for conditional generation
- **Curriculum Learning**: Phase 1 on silver data (~6300 rows), Phase 2 on gold data (~800 rows)
- **Evaluation**: Cosine similarity between sentence embeddings of generated vs target labels
- **Hardware**: Kaggle T4 x2, fp16 mixed precision

In [1]:
!pip install -q transformers datasets accelerate sentencepiece sentence-transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    get_cosine_schedule_with_warmup,
)
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Device: {device} | GPUs available: {n_gpus}")
if torch.cuda.is_available():
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Device: cuda | GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 1. Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — tweak these knobs
# ══════════════════════════════════════════════════════════════

MODEL_NAMES = [
    "google/flan-t5-large",
    "google/flan-t5-xl",
    "google/flan-t5-xxl",
]
EMBED_MODEL = "all-MiniLM-L6-v2"          # for cosine-similarity evaluation

# Paths (adjust for Kaggle: /kaggle/input/your-dataset-name/)
SILVER_PATH = "/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/taxonomy_pairs.csv"
GOLD_PATH   = "/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/verified_pairs.csv"

# Tokenization lengths
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 32

# Phase 1: Silver data training
P1_EPOCHS     = 5
P1_BATCH_SIZE = 16    # per device
P1_LR         = 3e-4
P1_WARMUP     = 0.06  # fraction of total steps
P1_WEIGHT_DECAY = 0.01

# Phase 2: Gold data fine-tuning
P2_EPOCHS     = 15
P2_BATCH_SIZE = 16
P2_LR         = 1e-4  # lower LR for refinement
P2_WARMUP     = 0.10
P2_WEIGHT_DECAY = 0.01

# Early stopping patience (number of eval steps without improvement)
PATIENCE = 3

# Output directories (will be created per model)
OUTPUT_BASE_DIR = "./flan_t5_models"

## 2. Load & Prepare Data

The prompt template encodes both the research field and keyword list into a natural instruction for the model.

In [4]:
# ── Load CSVs ──
silver_df = pd.read_csv(SILVER_PATH)
gold_df   = pd.read_csv(GOLD_PATH)

# Standardize column order
for df in [silver_df, gold_df]:
    assert set(df.columns) >= {"research_field", "topic_words", "topic_label"}, \
        f"Missing columns! Found: {df.columns.tolist()}"

silver_df = silver_df[["research_field", "topic_words", "topic_label"]].dropna().reset_index(drop=True)
gold_df   = gold_df[["research_field", "topic_words", "topic_label"]].dropna().reset_index(drop=True)

print(f"Silver dataset: {len(silver_df):,} rows")
print(f"Gold dataset:   {len(gold_df):,} rows")
silver_df.head(3)

Silver dataset: 6,347 rows
Gold dataset:   813 rows


,research_field,topic_words,topic_label
0,Legal Theory,"positivism, natural, rights, morality, rule",Natural Law vs Legal Positivism
1,Legal Theory,"jurisprudence, interpretation, adjudication, p...",Jurisprudential Theory and Interpretation
2,Legal Theory,"critical, race, gender, marxist, feminist",Critical Legal Theory


In [5]:
# ── Prompt template ──
PROMPT_TEMPLATE = (
    "Generate a concise topic label for the following.\n"
    "Research field: {field}\n"
    "Keywords: {words}\n"
    "Topic label:"
)

def build_prompt(row):
    return PROMPT_TEMPLATE.format(field=row["research_field"], words=row["topic_words"])

# Create input-target pairs
silver_df["input_text"]  = silver_df.apply(build_prompt, axis=1)
silver_df["target_text"] = silver_df["topic_label"]

gold_df["input_text"]  = gold_df.apply(build_prompt, axis=1)
gold_df["target_text"] = gold_df["topic_label"]

# ── Split gold data: 80% train, 20% validation ──
gold_train_df, gold_val_df = train_test_split(
    gold_df, test_size=0.20, random_state=SEED, shuffle=True
)
gold_train_df = gold_train_df.reset_index(drop=True)
gold_val_df   = gold_val_df.reset_index(drop=True)

# ── Split silver data: use 5% as held-out for Phase 1 validation ──
silver_train_df, silver_val_df = train_test_split(
    silver_df, test_size=0.05, random_state=SEED, shuffle=True
)
silver_train_df = silver_train_df.reset_index(drop=True)
silver_val_df   = silver_val_df.reset_index(drop=True)

print(f"Silver  train: {len(silver_train_df):,} | Silver  val: {len(silver_val_df):,}")
print(f"Gold    train: {len(gold_train_df):,} | Gold    val: {len(gold_val_df):,}")
print(f"\nSample prompt:\n{silver_df['input_text'].iloc[0]}")
print(f"Target: {silver_df['target_text'].iloc[0]}")

Silver  train: 6,029 | Silver  val: 318
Gold    train: 650 | Gold    val: 163

Sample prompt:
Generate a concise topic label for the following.
Research field: Legal Theory
Keywords: positivism, natural, rights, morality, rule
Topic label:
Target: Natural Law vs Legal Positivism


## 3. Tokenizer, Model & Dataset

In [ ]:
# All models will be loaded dynamically in the training function below

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model parameters: 247,577,856
Trainable:        247,577,856


In [ ]:
import os
from pathlib import Path

def train_model_pipeline(model_name: str, embed_model):
    """
    Train a single model on both silver and gold data phases.
    Returns (trained_model, trainer_p1, trainer_p2, final_output_dir, results_df, model_base_name)
    Uses global silver_train_df, silver_val_df, gold_train_df, gold_val_df
    """
    
    # ── Load tokenizer & model ──
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    # Create model-specific output dirs
    model_base_name = model_name.split("/")[-1]  # e.g., "flan-t5-large"
    p1_output_dir = os.path.join(OUTPUT_BASE_DIR, model_base_name, "phase1_silver")
    p2_output_dir = os.path.join(OUTPUT_BASE_DIR, model_base_name, "phase2_gold")
    final_model_dir = os.path.join(OUTPUT_BASE_DIR, model_base_name, "final_model")
    
    os.makedirs(p1_output_dir, exist_ok=True)
    os.makedirs(p2_output_dir, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"Training {model_name}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*70}\n")
    
    # ── Re-tokenize datasets with this model's tokenizer ──
    def tokenize_fn_for_model(examples):
        model_inputs = tokenizer(
            examples["input_text"],
            max_length=MAX_INPUT_LEN,
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=examples["target_text"],
            max_length=MAX_TARGET_LEN,
            truncation=True,
            padding=False,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    
    def make_hf_dataset_for_model(df):
        ds = HFDataset.from_pandas(df[["input_text", "target_text"]])
        ds = ds.map(tokenize_fn_for_model, batched=True, remove_columns=["input_text", "target_text"])
        return ds
    
    # Tokenize with current model's tokenizer
    silver_train_ds_model = make_hf_dataset_for_model(silver_train_df)
    silver_val_ds_model = make_hf_dataset_for_model(silver_val_df)
    gold_train_ds_model = make_hf_dataset_for_model(gold_train_df)
    gold_val_ds_model = make_hf_dataset_for_model(gold_val_df)
    
    # Create data collator for this model
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, pad_to_multiple_of=8)
    
    # ── Phase 1 ──
    p1_steps_per_epoch = len(silver_train_ds_model) // (P1_BATCH_SIZE * max(n_gpus, 1))
    p1_total_steps = p1_steps_per_epoch * P1_EPOCHS
    p1_eval_steps = max(p1_steps_per_epoch // 2, 1)
    
    def compute_metrics_for_model(eval_preds):
        preds_ids, labels_ids = eval_preds
        labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds_ids, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)
        decoded_preds = [p.strip() for p in decoded_preds]
        decoded_labels = [l.strip() for l in decoded_labels]
        cos_sim = mean_cosine_similarity(decoded_preds, decoded_labels)
        return {"cosine_similarity": cos_sim}
    
    p1_args = Seq2SeqTrainingArguments(
        output_dir=p1_output_dir,
        num_train_epochs=P1_EPOCHS,
        per_device_train_batch_size=P1_BATCH_SIZE,
        per_device_eval_batch_size=P1_BATCH_SIZE * 2,
        learning_rate=P1_LR,
        weight_decay=P1_WEIGHT_DECAY,
        warmup_steps=int(p1_total_steps * P1_WARMUP),
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        eval_strategy="steps",
        eval_steps=p1_eval_steps,
        save_strategy="steps",
        save_steps=p1_eval_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="cosine_similarity",
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LEN,
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=2,
        seed=SEED,
        label_smoothing_factor=0.1,
        gradient_accumulation_steps=1,
    )
    
    p1_trainer = Seq2SeqTrainer(
        model=model,
        args=p1_args,
        train_dataset=silver_train_ds_model,
        eval_dataset=silver_val_ds_model,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_for_model,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    
    print(f"  Phase 1: {p1_total_steps} total steps")
    p1_trainer.train()
    print(f"  Phase 1 complete!")
    
    # ── Phase 2 ──
    p2_steps_per_epoch = len(gold_train_ds_model) // (P2_BATCH_SIZE * max(n_gpus, 1))
    p2_total_steps = p2_steps_per_epoch * P2_EPOCHS
    p2_eval_steps = max(p2_steps_per_epoch, 1)
    
    p2_args = Seq2SeqTrainingArguments(
        output_dir=p2_output_dir,
        num_train_epochs=P2_EPOCHS,
        per_device_train_batch_size=P2_BATCH_SIZE,
        per_device_eval_batch_size=P2_BATCH_SIZE * 2,
        learning_rate=P2_LR,
        weight_decay=P2_WEIGHT_DECAY,
        warmup_steps=int(p2_total_steps * P2_WARMUP),
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        eval_strategy="steps",
        eval_steps=p2_eval_steps,
        save_strategy="steps",
        save_steps=p2_eval_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="cosine_similarity",
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LEN,
        logging_steps=10,
        report_to="none",
        dataloader_num_workers=2,
        seed=SEED,
        label_smoothing_factor=0.05,
        gradient_accumulation_steps=1,
    )
    
    p2_trainer = Seq2SeqTrainer(
        model=model,
        args=p2_args,
        train_dataset=gold_train_ds_model,
        eval_dataset=gold_val_ds_model,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_for_model,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )
    
    print(f"  Phase 2: {p2_total_steps} total steps")
    p2_trainer.train()
    print(f"  Phase 2 complete!")
    
    # ── Save final model ──
    model.save_pretrained(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)
    print(f"  Model saved to {final_model_dir}/")
    
    # ── Generate and evaluate on gold validation ──
    @torch.no_grad()
    def generate_labels_for_model(input_texts: list[str], batch_size: int = 32) -> list[str]:
        model.eval()
        all_preds = []
        for i in range(0, len(input_texts), batch_size):
            batch = input_texts[i : i + batch_size]
            inputs = tokenizer(
                batch,
                max_length=MAX_INPUT_LEN,
                truncation=True,
                padding=True,
                return_tensors="pt",
            ).to(model.device)
            outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_preds.extend([d.strip() for d in decoded])
        return all_preds
    
    val_inputs = gold_val_df["input_text"].tolist()
    val_targets = gold_val_df["target_text"].tolist()
    val_preds = generate_labels_for_model(val_inputs)
    
    # Compute cosine similarities
    pred_embs = embed_model.encode(val_preds, batch_size=128, convert_to_numpy=True)
    target_embs = embed_model.encode(val_targets, batch_size=128, convert_to_numpy=True)
    per_sample_sim = np.array([
        sk_cosine(p.reshape(1, -1), t.reshape(1, -1))[0, 0]
        for p, t in zip(pred_embs, target_embs)
    ])
    
    results_df = pd.DataFrame({
        "model": model_name,
        "research_field": gold_val_df["research_field"].values,
        "keywords": gold_val_df["topic_words"].values,
        "target_label": val_targets,
        "predicted_label": val_preds,
        "cosine_sim": per_sample_sim.round(4),
    })
    
    print(f"\n  Final Results ({model_base_name}):")
    print(f"    Mean Cosine Similarity:   {per_sample_sim.mean():.4f}")
    print(f"    % samples ≥ 0.8:         {(per_sample_sim >= 0.8).mean() * 100:.1f}%")
    print(f"    % samples ≥ 0.9:         {(per_sample_sim >= 0.9).mean() * 100:.1f}%\n")
    
    return model, p1_trainer, p2_trainer, final_model_dir, results_df, model_base_name

## 4. Evaluation Metric — Cosine Similarity

We load a sentence-transformer once and use it throughout training and final evaluation.

In [8]:
# ── Sentence embedding model (for evaluation only) ──
embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")  # keep off GPU to save VRAM

def mean_cosine_similarity(preds: list[str], targets: list[str]) -> float:
    """Compute mean pairwise cosine similarity between two lists of strings."""
    pred_embs   = embed_model.encode(preds,   batch_size=128, show_progress_bar=False, convert_to_numpy=True)
    target_embs = embed_model.encode(targets, batch_size=128, show_progress_bar=False, convert_to_numpy=True)
    # Row-wise cosine similarity
    sims = np.array([
        sk_cosine(p.reshape(1, -1), t.reshape(1, -1))[0, 0]
        for p, t in zip(pred_embs, target_embs)
    ])
    return float(sims.mean())

def compute_metrics(eval_preds):
    """
    Called by Seq2SeqTrainer during evaluation.
    eval_preds = (predictions_ids, label_ids)
    """
    preds_ids, labels_ids = eval_preds

    # Replace -100 (ignore index) with pad token for decoding
    labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds_ids,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    cos_sim = mean_cosine_similarity(decoded_preds, decoded_labels)
    return {"cosine_similarity": cos_sim}

# Quick sanity check
test_sim = mean_cosine_similarity(
    ["Natural Language Processing", "Computer Vision"],
    ["NLP and Text Mining", "Image Recognition"]
)
print(f"Sanity check cosine sim: {test_sim:.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sanity check cosine sim: 0.7075


## 5. Train All 3 Models

Run the training pipeline for each model (flan-t5-large, flan-t5-XL, flan-t5-XXL). Each model will be trained on silver data (Phase 1) then gold data (Phase 2), and saved separately.

In [ ]:
# ── Train all 3 models sequentially ──
trained_models = {}
all_results = []

for model_name in MODEL_NAMES:
    model, p1_trainer, p2_trainer, final_dir, results_df, model_base_name = train_model_pipeline(
        model_name=model_name,
        embed_model=embed_model
    )
    
    trained_models[model_base_name] = {
        'model': model,
        'final_dir': final_dir,
        'p1_trainer': p1_trainer,
        'p2_trainer': p2_trainer,
    }
    all_results.append(results_df)

print(f"\n{'='*70}")
print("✓ All 3 models trained and saved!")
print(f"{'='*70}")

## 6. Evaluation Results — All Models

In [ ]:
# ── Combine and compare results from all 3 models ──
combined_results = pd.concat(all_results, ignore_index=True)

# Summary statistics per model
print(f"\n{'='*80}")
print("SUMMARY: Cosine Similarity Performance Across All Models")
print(f"{'='*80}\n")

summary_stats = combined_results.groupby('model')['cosine_sim'].agg([
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std Dev', 'std'),
    ('Min', 'min'),
    ('Max', 'max'),
    ('% ≥ 0.8', lambda x: (x >= 0.8).mean() * 100),
    ('% ≥ 0.9', lambda x: (x >= 0.9).mean() * 100),
]).round(4)

print(summary_stats)

# Show best and worst predictions for each model
print(f"\n{'='*80}")
print("BEST PREDICTIONS BY MODEL")
print(f"{'='*80}\n")

for model_name in MODEL_NAMES:
    model_results = combined_results[combined_results['model'] == model_name]
    print(f"\n{model_name}:")
    print(model_results.nlargest(3, 'cosine_sim')[['target_label', 'predicted_label', 'cosine_sim']].to_string(index=False))

print(f"\n{'='*80}")
print("WORST PREDICTIONS BY MODEL")
print(f"{'='*80}\n")

for model_name in MODEL_NAMES:
    model_results = combined_results[combined_results['model'] == model_name]
    print(f"\n{model_name}:")
    print(model_results.nsmallest(3, 'cosine_sim')[['target_label', 'predicted_label', 'cosine_sim']].to_string(index=False))

In [ ]:
# ── Distribution comparison plots ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot across all models
colors = ['steelblue', 'coral', 'seagreen']
model_names_short = [m.split("/")[-1] for m in MODEL_NAMES]
boxplot_data = [combined_results[combined_results['model'] == m]['cosine_sim'].values for m in MODEL_NAMES]
bp = axes[0].boxplot(boxplot_data, labels=model_names_short, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].axhline(0.8, color='orange', linestyle='--', alpha=0.5, label='0.8 threshold')
axes[0].axhline(0.9, color='green', linestyle='--', alpha=0.5, label='0.9 threshold')
axes[0].set_ylabel('Cosine Similarity')
axes[0].set_title('Cosine Similarity Distribution by Model')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Histogram overlay
for model_name, color in zip(MODEL_NAMES, colors):
    model_data = combined_results[combined_results['model'] == model_name]['cosine_sim']
    axes[1].hist(model_data, bins=20, alpha=0.5, label=model_name.split("/")[-1], color=color)
axes[1].axvline(0.8, color='orange', linestyle='--', alpha=0.7, label='0.8 threshold')
axes[1].axvline(0.9, color='green', linestyle='--', alpha=0.7, label='0.9 threshold')
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Cosine Similarities')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Inference — Compare All 3 Models

Generate predictions for example inputs using all three fine-tuned models.

In [ ]:
# ── Load all 3 models and generate inference examples ──

def predict_topic_label_for_model(model_dir: str, research_field: str, keywords: str) -> str:
    """Generate a topic label using a specific fine-tuned model."""
    tokenizer_inf = AutoTokenizer.from_pretrained(model_dir)
    model_inf = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device)
    model_inf.eval()
    
    prompt = PROMPT_TEMPLATE.format(field=research_field, words=keywords)
    inputs = tokenizer_inf(prompt, max_length=MAX_INPUT_LEN, truncation=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output = model_inf.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    
    prediction = tokenizer_inf.decode(output[0], skip_special_tokens=True).strip()
    del model_inf
    torch.cuda.empty_cache()
    return prediction

# ── Example predictions ──
examples = [
    ("Machine Learning", "neural, network, deep, learning, convolutional"),
    ("Economics", "inflation, monetary, policy, central, bank"),
    ("Biology", "gene, expression, regulation, transcription, protein"),
    ("Computer Science", "graph, algorithm, optimization, complexity, polynomial"),
]

print(f"\n{'='*120}")
print("INFERENCE EXAMPLES — Comparing All 3 Models")
print(f"{'='*120}\n")

for field, keywords in examples:
    print(f"Research Field: {field}")
    print(f"Keywords: {keywords}\n")
    
    for model_name in MODEL_NAMES:
        model_base_name = model_name.split("/")[-1]
        model_dir = os.path.join(OUTPUT_BASE_DIR, model_base_name, "final_model")
        prediction = predict_topic_label_for_model(model_dir, field, keywords)
        print(f"  {model_base_name:20s} → {prediction}")
    
    print()

## 8. Export Results

In [ ]:
# ── Save evaluation results for all models ──
combined_results.to_csv("evaluation_results_all_models.csv", index=False)
print(f"✓ Combined evaluation results saved to evaluation_results_all_models.csv")

# Save individual model results
for model_name in MODEL_NAMES:
    model_base_name = model_name.split("/")[-1]
    model_results = combined_results[combined_results['model'] == model_name]
    csv_path = f"evaluation_results_{model_base_name}.csv"
    model_results.to_csv(csv_path, index=False)
    print(f"✓ {model_base_name} results saved to {csv_path}")

print(f"\n{'='*70}")
print("TRAINING SUMMARY")
print(f"{'='*70}")

training_summary = []
for model_name in MODEL_NAMES:
    model_base_name = model_name.split("/")[-1]
    model_results = combined_results[combined_results['model'] == model_name]
    
    training_summary.append({
        'Model': model_name,
        'Parameters': 'large (770M)' if 'large' in model_base_name else 'XL (3B)' if 'xl' in model_base_name else 'XXL (11B)',
        'Mean Cosine Sim': f"{model_results['cosine_sim'].mean():.4f}",
        'Median Cosine Sim': f"{model_results['cosine_sim'].median():.4f}",
        '% ≥ 0.8': f"{(model_results['cosine_sim'] >= 0.8).mean() * 100:.1f}%",
        '% ≥ 0.9': f"{(model_results['cosine_sim'] >= 0.9).mean() * 100:.1f}%",
        'Model Path': os.path.join(OUTPUT_BASE_DIR, model_base_name, "final_model"),
    })

summary_table = pd.DataFrame(training_summary)
print(summary_table.to_string(index=False))

print(f"\n{'='*70}")
print("✓ All 3 models successfully trained and saved!")
print(f"{'='*70}")

Evaluation results saved to evaluation_results.csv (163 rows)

TRAINING SUMMARY
Model:               google/flan-t5-base
Silver data:         6,347 rows (5 epochs)
Gold data:           813 rows (15 epochs)
Phase 1 best metric: 0.7129 (silver val cos-sim)
Phase 2 best metric: 0.6557 (gold val cos-sim)
Final mean cos-sim:  0.6607 (gold val)
Model saved to:      ./final_model/
